# Migliorare un modello partendo dagli eval

Notebook di supporto all'articolo [Post-training in pratica: migliorare un modello partendo dagli eval](https://bigghis.github.io/posts/POST-TRAINING-EVAL-LAB/) della serie sul **post-training** degli LLM.

Mettiamo in pratica un ciclo completo di miglioramento guidato dagli eval, su problemi di matematica di [GSM8K](https://huggingface.co/datasets/openai/gsm8k):

1. **Valutare** il modello e misurare l'accuratezza di partenza
2. **Classificare** gli errori per tipo
3. **Raggruppare** gli errori con embedding e k-means per trovare schemi ricorrenti
4. **Selezionare** dal train esempi simili agli errori (dati mirati)
5. **Fine-tuning** con LoRA sui dati mirati
6. **Rivalutare** e capire se il miglioramento è reale o solo rumore

## Due modalità

- `BACKEND=demo` (predefinito): nessun LLM da scaricare, gira su CPU (anche su Binder). Le risposte del modello sono **simulate**, ma tutta la pipeline di analisi (valutazione, classificazione, clustering, selezione dei dati, statistiche) è reale.
- `BACKEND=hf`: usa un modello vero con Hugging Face `transformers` e fa davvero il fine-tuning. Serve una GPU (per esempio Colab).

## Setup

In [ ]:
# Su Colab, per il backend "hf":
# !pip install -q transformers peft accelerate datasets sentence-transformers scikit-learn scipy

import json
import math
import os
import random
import re
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

BACKEND = os.environ.get("POSTTRAINING_BACKEND", "demo")         # "demo" oppure "hf"
MODEL_NAME = os.environ.get("MODEL_NAME", "Qwen/Qwen2.5-0.5B")    # modello base da migliorare
JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "")                   # es. "Qwen/Qwen2.5-1.5B-Instruct"; vuoto = regole
EMBEDDINGS = os.environ.get("EMBEDDINGS", "auto")                 # "auto", "sentence-transformers" o "tfidf"

N_DEV = int(os.environ.get("N_DEV", 300))            # domande per l'error analysis
N_HOLDOUT = int(os.environ.get("N_HOLDOUT", 300))    # domande mai usate per decidere: solo misura finale
BUDGET_PER_CATEGORY = int(os.environ.get("BUDGET_PER_CATEGORY", 60))
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", 320))
NUM_EPOCHS = float(os.environ.get("NUM_EPOCHS", 2))

SEED = 42
OUT_DIR = "lab_results"
os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)

print(f"Backend: {BACKEND} | modello: {MODEL_NAME} | dev: {N_DEV} | holdout: {N_HOLDOUT}")

## 1. I dati: dev set e holdout

Dal test set di GSM8K ricaviamo **due** insiemi separati:

- il **dev set** serve per l'error analysis e per scegliere i dati di training;
- l'**holdout** non viene mai guardato per prendere decisioni: serve solo a misurare, alla fine, se il miglioramento è reale.

Se usassimo le stesse domande sia per scegliere i dati sia per misurare il risultato, rischieremmo di "insegnare per il test".

In [ ]:
from datasets import load_dataset

gsm8k = load_dataset("openai/gsm8k", "main")
train = gsm8k["train"]
test = gsm8k["test"].shuffle(seed=SEED)

dev_set = test.select(range(N_DEV))
holdout_set = test.select(range(N_DEV, N_DEV + N_HOLDOUT))

print(f"train: {len(train)} | dev: {len(dev_set)} | holdout: {len(holdout_set)}")
print("\nEsempio:\n", dev_set[0]["question"], "\n---\n", dev_set[0]["answer"])

## 2. Il verifier: estrarre e confrontare le risposte

Nelle soluzioni di GSM8K la risposta finale segue `####`. Dalle risposte del modello la estraiamo in modo robusto: prima `####`, poi frasi come *"The answer is 42"*, infine l'ultimo numero del testo.

In [ ]:
NUM = r"-?\d[\d,]*(?:\.\d+)?"


def to_float(s):
    try:
        return float(s.replace(",", "").rstrip("."))
    except (ValueError, AttributeError):
        return None


def extract_gold(answer):
    """Risposta corretta di GSM8K: il numero dopo ####."""
    return to_float(answer.split("####")[-1].strip())


def extract_prediction(text):
    """Risposta del modello: ####, poi 'the answer is', poi l'ultimo numero."""
    m = re.search(r"####\s*\$?\s*(" + NUM + ")", text)
    if m:
        return to_float(m.group(1))
    m = re.search(r"answer is\s*:?\s*\$?\s*(" + NUM + ")", text, re.IGNORECASE)
    if m:
        return to_float(m.group(1))
    numbers = re.findall(NUM, text)
    return to_float(numbers[-1]) if numbers else None


def is_correct(pred, gold):
    return pred is not None and gold is not None and abs(pred - gold) < 1e-3


assert extract_gold("... 48/2 = <<48/2=24>>24\n#### 1,024") == 1024.0
assert extract_prediction("So she has 5 apples.\n#### 5") == 5.0
assert extract_prediction("The answer is $1,200.") == 1200.0
assert extract_prediction("3 + 4 = 7, then 7 * 2 = 14") == 14.0
assert extract_prediction("I don't know") is None
print("verifier ok")

## 3. Generare le risposte

Il prompt ha sempre lo stesso formato, `Question: ...\nAnswer:`, che useremo identico anche nel fine-tuning. La generazione è **greedy** (deterministica), così rilanciando la valutazione si ottengono gli stessi risultati.

Nel backend `demo` le risposte sono **simulate**: il "modello" sbaglia più spesso sui problemi con percentuali e frazioni e su quelli lunghi, così che l'error analysis abbia degli schemi da scoprire.

In [ ]:
def build_prompt(question):
    return f"Question: {question}\nAnswer:"


def clean_response(text):
    """Un modello base, finita la risposta, tende a inventare la domanda successiva: la tagliamo."""
    for stop in ("\nQuestion:", "[Question]", "\n\n\n"):
        text = text.split(stop)[0]
    return text.strip()


def strip_calculator(answer):
    """Toglie le annotazioni <<48/2=24>> delle soluzioni di GSM8K."""
    return re.sub(r"<<[^>]*>>", "", answer)


# ---------- backend "demo": risposte simulate ----------

PERCENT_WORDS = ("%", "percent", "half", "third", "quarter", "fraction")


def simulate_response(example, rng):
    q, solution = example["question"], strip_calculator(example["answer"])
    gold = extract_gold(example["answer"])
    p_error = 0.30
    hard_math = any(w in q.lower() for w in PERCENT_WORDS)
    long_problem = len(q) > 300
    p_error += 0.35 * hard_math + 0.20 * long_problem
    if rng.random() > p_error:
        return solution                                   # risposta corretta

    body = solution.split("####")[0].strip()
    lines = body.split("\n")
    r = rng.random()
    if r < 0.10:                                          # formato: niente numero dopo ####
        return "\n".join(lines[:-1] or lines) + "\n#### the answer is above"
    if r < 0.25:                                          # incompleta: si interrompe a metà
        return "\n".join(lines[: max(1, len(lines) // 2)])
    if hard_math and r < 0.75 or r < 0.45:                # calcolo: risultato finale un po' sbagliato
        wrong = gold + rng.choice([-1, 1]) * max(1, round(abs(gold) * rng.uniform(0.02, 0.2)))
    else:                                                 # ragionamento: risultato molto diverso
        wrong = gold * rng.choice([2, 3, 0.5]) + rng.randint(5, 40)
    wrong = round(wrong, 2)
    wrong_str = str(int(wrong)) if float(wrong).is_integer() else str(wrong)
    gold_str = str(int(gold)) if float(gold).is_integer() else str(gold)
    lines[-1] = lines[-1].replace(gold_str, wrong_str)    # l'errore compare anche nei passaggi
    return "\n".join(lines) + f"\n#### {wrong_str}"


def simulate_responses(dataset, seed):
    rng = random.Random(seed)
    return [simulate_response(ex, rng) for ex in dataset]


# ---------- backend "hf": modello vero ----------

model = tokenizer = None
if BACKEND == "hf":
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    torch.manual_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else (
        torch.float16 if device == "cuda" else torch.float32)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).to(device)
    model.eval()
    print(f"Caricato {MODEL_NAME} su {device} ({dtype})")


def generate_responses(model, tokenizer, questions, batch_size=16, max_new_tokens=MAX_NEW_TOKENS):
    import torch

    tokenizer.padding_side = "left"                       # per generare il padding va a sinistra
    responses = []
    for i in tqdm(range(0, len(questions), batch_size), desc="generazione"):
        prompts = [build_prompt(q) for q in questions[i:i + batch_size]]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
        new_tokens = out[:, enc["input_ids"].shape[1]:]
        responses += [clean_response(t) for t in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)]
    return responses

## 4. Valutare il modello

`evaluate_model` genera le risposte, estrae il numero finale, lo confronta con quello corretto e salva tutto in un dizionario che useremo nei passi successivi.

In [ ]:
def evaluate_model(dataset, name, responses=None):
    questions = [ex["question"] for ex in dataset]
    golds = [extract_gold(ex["answer"]) for ex in dataset]
    if responses is None:
        responses = generate_responses(model, tokenizer, questions)
    preds = [extract_prediction(r) for r in responses]
    correct = [is_correct(p, g) for p, g in zip(preds, golds)]
    results = {
        "name": name,
        "questions": questions,
        "golds": golds,
        "responses": responses,
        "predictions": preds,
        "correct": correct,
        "accuracy": float(np.mean(correct)),
    }
    print(f"{name}: accuratezza {results['accuracy']:.3f} ({sum(correct)}/{len(correct)})")
    return results


def save_results(results, filename):
    with open(os.path.join(OUT_DIR, filename), "w") as f:
        json.dump(results, f, indent=2)


if BACKEND == "demo":
    dev_before = evaluate_model(dev_set, "dev, prima", simulate_responses(dev_set, seed=1))
    holdout_before = evaluate_model(holdout_set, "holdout, prima", simulate_responses(holdout_set, seed=2))
else:
    dev_before = evaluate_model(dev_set, "dev, prima")
    holdout_before = evaluate_model(holdout_set, "holdout, prima")

save_results(dev_before, "dev_before.json")
save_results(holdout_before, "holdout_before.json")

In [ ]:
wrong_idx = [i for i, c in enumerate(dev_before["correct"]) if not c]
print(f"{len(wrong_idx)} errori sul dev set. Qualche esempio:\n")
for i in wrong_idx[:3]:
    print("DOMANDA:  ", dev_before["questions"][i][:150], "...")
    print("CORRETTA: ", dev_before["golds"][i], "| PREDETTA:", dev_before["predictions"][i])
    print("RISPOSTA: ", dev_before["responses"][i][-200:].replace("\n", " "), "\n")

## 5. Classificare gli errori

Ogni errore va in una di cinque categorie. Se è configurato un modello giudice (`JUDGE_MODEL`), la classificazione la fa un LLM; altrimenti si usano regole semplici.

Due accorgimenti importanti per il giudice LLM:

- il prompt include i **passaggi** della risposta del modello, non solo il numero finale, altrimenti il giudice non può distinguere un errore di calcolo da uno di ragionamento;
- la risposta del giudice si interpreta in modo **robusto**, cercando il nome esatto della categoria (anche scritto con lo spazio) e ricadendo su `other`.

In [ ]:
ERROR_CATEGORIES = [
    "calculation_error",    # errori di calcolo
    "reasoning_error",      # errori di ragionamento
    "incomplete_solution",  # risposta parziale o non finita
    "format_error",         # risposta nel formato sbagliato
    "other",                # altro
]


def classify_by_rules(question, gold, response, pred):
    has_marker = "####" in response
    marker_number = re.search(r"####\s*\$?\s*" + NUM, response)
    if pred is None or (has_marker and not marker_number):
        return "format_error"
    if not has_marker and not re.search(r"answer is", response, re.IGNORECASE):
        return "incomplete_solution"
    if abs(pred - gold) / max(abs(gold), 1) <= 0.25:
        return "calculation_error"
    return "reasoning_error"


def parse_category(text):
    text = text.strip().lower()
    for category in ERROR_CATEGORIES:
        if category in text or category.replace("_", " ") in text:
            return category
    return "other"


judge = None
if BACKEND == "hf" and JUDGE_MODEL:
    from transformers import pipeline
    judge = pipeline("text-generation", model=JUDGE_MODEL, torch_dtype="auto", device_map="auto")


def classify_by_llm(question, gold, response, pred):
    prompt = f"""Analyze this math error and categorize it.

PROBLEM: {question}
CORRECT ANSWER: {gold}
MODEL SOLUTION: {response[-1500:]}
PREDICTED ANSWER: {pred}

Choose ONE category: {', '.join(ERROR_CATEGORIES)}
Respond with just the category name."""
    out = judge([{"role": "user", "content": prompt}], max_new_tokens=10, do_sample=False)
    return parse_category(out[0]["generated_text"][-1]["content"])


def analyze_errors(results):
    classify = classify_by_llm if judge is not None else classify_by_rules
    records = []
    for i, ok in enumerate(results["correct"]):
        if ok:
            continue
        args = (results["questions"][i], results["golds"][i], results["responses"][i], results["predictions"][i])
        records.append({"index": i, "question": args[0], "category": classify(*args)})
    return records


dev_errors = analyze_errors(dev_before)
counts = Counter(r["category"] for r in dev_errors)
print("Classificazione:", "LLM" if judge is not None else "regole")
for category, n in counts.most_common():
    print(f"  {category:20s} {n:4d}  ({100 * n / len(dev_errors):.1f}%)")
save_results(dev_errors, "dev_errors.json")

## 6. Raggruppare gli errori

Trasformiamo le domande sbagliate in **embedding** e le raggruppiamo con il **k-means**. Il numero di gruppi lo scegliamo provando alcuni valori e guardando il **silhouette score** (più alto = gruppi più compatti e separati).

Gli embedding sono normalizzati, quindi il prodotto scalare tra due embedding è la loro similarità del coseno. Se `sentence-transformers` non è installato, si ripiega su TF-IDF + SVD, meno preciso ma senza modelli da scaricare.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score


def make_embedder():
    if EMBEDDINGS in ("auto", "sentence-transformers"):
        try:
            from sentence_transformers import SentenceTransformer
            st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
            print("Embedding: all-MiniLM-L6-v2")
            return lambda texts: st_model.encode(list(texts), batch_size=64, normalize_embeddings=True,
                                                 show_progress_bar=False)
        except ImportError:
            if EMBEDDINGS == "sentence-transformers":
                raise
    from sklearn.decomposition import TruncatedSVD
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.preprocessing import normalize

    print("Embedding: TF-IDF + SVD (fallback)")
    vectorizer = TfidfVectorizer(stop_words="english", min_df=2).fit(train["question"])
    svd = TruncatedSVD(n_components=128, random_state=SEED).fit(vectorizer.transform(train["question"]))
    return lambda texts: normalize(svd.transform(vectorizer.transform(list(texts))))


embed = make_embedder()
error_embeddings = embed([r["question"] for r in dev_errors])

scores = {}
for k in range(2, min(9, len(dev_errors))):           # servono più errori che gruppi
    labels = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit_predict(error_embeddings)
    scores[k] = silhouette_score(error_embeddings, labels)
best_k = max(scores, key=scores.get) if scores else 1
print("silhouette:", {k: round(v, 3) for k, v in scores.items()}, "-> k =", best_k)

kmeans = KMeans(n_clusters=best_k, random_state=SEED, n_init=10)
cluster_labels = kmeans.fit_predict(error_embeddings)
for record, label in zip(dev_errors, cluster_labels):
    record["cluster"] = int(label)

In [ ]:
df_errors = pd.DataFrame(dev_errors)
print("Gruppi (righe) incrociati con le categorie di errore (colonne):\n")
print(pd.crosstab(df_errors["cluster"], df_errors["category"], margins=True))

for cluster_id, group in df_errors.groupby("cluster"):
    print(f"\nGruppo {cluster_id} ({len(group)} errori), categoria prevalente: "
          f"{group['category'].mode()[0]}")
    for q in group["question"].head(3):
        print("  -", q[:110])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

counts = Counter(r["category"] for r in dev_errors)
axes[0].bar(counts.keys(), counts.values(), color="#2563eb")
axes[0].set_title("Errori per categoria")
axes[0].tick_params(axis="x", rotation=30)

sizes = Counter(cluster_labels)
axes[1].bar([str(k) for k in sorted(sizes)], [sizes[k] for k in sorted(sizes)], color="#ea580c")
axes[1].set_title("Errori per gruppo (k-means)")
axes[1].set_xlabel("gruppo")

points = PCA(n_components=2, random_state=SEED).fit_transform(error_embeddings)
sc = axes[2].scatter(points[:, 0], points[:, 1], c=cluster_labels, cmap="tab10", s=18)
axes[2].set_title("Errori proiettati in 2D (PCA)")
plt.colorbar(sc, ax=axes[2], label="gruppo")
plt.tight_layout()
plt.show()

## 7. Selezionare dati di training mirati

Per ogni categoria di errore cerchiamo nel **train** di GSM8K le domande più simili a quelle sbagliate, e le usiamo come dati di fine-tuning.

Tre accorgimenti:

- ogni errore contribuisce con **almeno un** esempio, anche quando gli errori di una categoria sono più del budget;
- ogni categoria ha un **budget** massimo di esempi, e si tengono i più simili;
- si **escludono** gli esempi di train quasi identici a una domanda di dev o di holdout (similarità > 0,95), per non far trapelare il test nel training.

Prepariamo anche una **selezione casuale** della stessa dimensione: è il confronto onesto per capire se i dati mirati servono davvero.

In [ ]:
def select_targeted_examples(errors, train, embed, budget_per_category, eval_questions,
                             max_sim_to_eval=0.95):
    train_questions = list(train["question"])
    train_emb = embed(train_questions)

    # 1. escludo i quasi-duplicati delle domande di valutazione
    eval_emb = embed(eval_questions)
    sim_to_eval = (train_emb @ eval_emb.T).max(axis=1)
    allowed = sim_to_eval < max_sim_to_eval
    print(f"esclusi {int((~allowed).sum())} esempi di train troppo simili al test")

    by_category = defaultdict(list)
    for e in errors:
        by_category[e["category"]].append(e["question"])

    selected = {}                                   # indice nel train -> (similarità, categoria)
    for category, questions in by_category.items():
        sims = embed(questions) @ train_emb.T       # (errori, train)
        sims[:, ~allowed] = -1.0
        per_error = max(1, math.ceil(budget_per_category / len(questions)))   # mai zero
        candidates = {}
        for row in sims:
            for j in np.argsort(row)[::-1][:per_error]:
                candidates[int(j)] = max(candidates.get(int(j), -1.0), float(row[j]))
        best = sorted(candidates.items(), key=lambda kv: -kv[1])[:budget_per_category]
        for j, s in best:
            if j not in selected or s > selected[j][0]:
                selected[j] = (s, category)
        print(f"{category:20s} {len(questions):4d} errori -> {len(best):3d} esempi "
              f"(similarità media {np.mean([s for _, s in best]):.2f})")

    return [{"question": train[j]["question"], "answer": train[j]["answer"],
             "category": cat, "similarity": s} for j, (s, cat) in selected.items()]


eval_questions = list(dev_set["question"]) + list(holdout_set["question"])
targeted = select_targeted_examples(dev_errors, train, embed, BUDGET_PER_CATEGORY, eval_questions)

rng = random.Random(SEED)
random_baseline = [train[j] for j in rng.sample(range(len(train)), len(targeted))]
print(f"\n{len(targeted)} esempi mirati, {len(random_baseline)} esempi casuali per il confronto")

example = max(targeted, key=lambda e: e["similarity"])
print(f"\nEsempio ({example['category']}, similarità {example['similarity']:.2f}):\n{example['question'][:200]}")
save_results(targeted, "targeted_examples.json")

## 8. Fine-tuning con LoRA

Due scelte rispetto a un fine-tuning "ingenuo":

- **loss mask**: la loss si calcola solo sui token della risposta, non su quelli della domanda (le etichette della domanda valgono `-100`);
- **LoRA** invece del full fine-tuning: si addestra una piccola frazione dei parametri, con un learning rate adatto (`2e-4`), e il modello entra anche in GPU piccole.

Nel backend `demo` il fine-tuning non viene eseguito.

In [ ]:
def build_sft_dataset(examples, tokenizer, max_length=512):
    from datasets import Dataset

    rows = []
    for ex in examples:
        prompt_ids = tokenizer(build_prompt(ex["question"]) + " ", add_special_tokens=False)["input_ids"]
        answer_ids = tokenizer(strip_calculator(ex["answer"]) + tokenizer.eos_token,
                               add_special_tokens=False)["input_ids"]
        input_ids = (prompt_ids + answer_ids)[:max_length]
        labels = ([-100] * len(prompt_ids) + answer_ids)[:max_length]      # loss mask
        rows.append({"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels})
    return Dataset.from_list(rows)


def finetune_lora(examples, run_name):
    import torch
    from peft import LoraConfig, get_peft_model
    from transformers import (AutoModelForCausalLM, DataCollatorForSeq2Seq, Trainer,
                              TrainingArguments)

    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=model.dtype).to(model.device)
    base.config.use_cache = False
    peft_model = get_peft_model(base, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM"))
    peft_model.print_trainable_parameters()

    shuffled = random.Random(SEED).sample(examples, len(examples))
    n_val = max(1, len(shuffled) // 10)
    tokenizer.padding_side = "right"
    train_ds = build_sft_dataset(shuffled[n_val:], tokenizer)
    val_ds = build_sft_dataset(shuffled[:n_val], tokenizer)

    on_gpu = torch.cuda.is_available()
    total_steps = math.ceil(len(train_ds) / 16) * NUM_EPOCHS         # 16 = batch 4 x accumulo 4
    args = TrainingArguments(
        output_dir=os.path.join(OUT_DIR, run_name),
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=max(1, int(0.05 * total_steps)),
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=10,
        save_strategy="no",
        report_to=[],
        bf16=on_gpu and torch.cuda.is_bf16_supported(),
        fp16=on_gpu and not torch.cuda.is_bf16_supported(),
        seed=SEED,
    )
    trainer = Trainer(model=peft_model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=DataCollatorForSeq2Seq(tokenizer, label_pad_token_id=-100, padding=True))
    trainer.train()
    peft_model.config.use_cache = True
    peft_model.eval()
    return peft_model, trainer.state.log_history


def plot_losses(log_history, title):
    train_logs = [(l["step"], l["loss"]) for l in log_history if "loss" in l]
    eval_logs = [(l["step"], l["eval_loss"]) for l in log_history if "eval_loss" in l]
    plt.figure(figsize=(8, 4))
    if train_logs:
        plt.plot(*zip(*train_logs), marker="o", label="training loss")
    if eval_logs:
        plt.plot(*zip(*eval_logs), marker="s", label="validation loss")
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


if BACKEND == "hf":
    tuned_model, log_history = finetune_lora(targeted, "lora_targeted")
    plot_losses(log_history, "Fine-tuning sui dati mirati")
else:
    print("Backend demo: fine-tuning saltato.")

## 9. Rivalutare

Rivalutiamo il modello su dev e holdout. Il numero che conta di più è quello sull'**holdout**, perché su quelle domande non abbiamo preso nessuna decisione.

Nel backend `demo` i risultati dopo il fine-tuning sono **simulati** (una parte degli errori viene corretta, qualche risposta giusta diventa sbagliata), solo per mostrare come si leggono i confronti.

In [ ]:
def simulate_after(before, fix_rate, break_rate, seed):
    rng = random.Random(seed)
    after = dict(before, name=before["name"].replace("prima", "dopo (simulato)"))
    after["correct"] = [(rng.random() > break_rate) if ok else (rng.random() < fix_rate)
                        for ok in before["correct"]]
    after["predictions"] = [g if ok else p for ok, g, p in
                            zip(after["correct"], before["golds"], before["predictions"])]
    after["accuracy"] = float(np.mean(after["correct"]))
    print(f"{after['name']}: accuratezza {after['accuracy']:.3f}")
    return after


if BACKEND == "hf":
    model = tuned_model
    dev_after = evaluate_model(dev_set, "dev, dopo")
    holdout_after = evaluate_model(holdout_set, "holdout, dopo")
else:
    dev_after = simulate_after(dev_before, fix_rate=0.30, break_rate=0.04, seed=3)
    holdout_after = simulate_after(holdout_before, fix_rate=0.20, break_rate=0.04, seed=4)

save_results(dev_after, "dev_after.json")
save_results(holdout_after, "holdout_after.json")

## 10. Il miglioramento è reale?

Per ogni domanda confrontiamo prima e dopo:

- **ancora corrette** e **ancora sbagliate** non dicono nulla sulla differenza;
- **nuove corrette** e **nuove sbagliate** sono le domande che hanno cambiato esito.

Il **test di McNemar** si chiede se le nuove corrette sono davvero più delle nuove sbagliate o se la differenza può essere caso (è un test binomiale sulle sole domande che hanno cambiato esito). Il **bootstrap** dà un intervallo di confidenza per la differenza di accuratezza.

In [ ]:
from scipy.stats import binomtest


def paired_bootstrap(correct_after, correct_before, n_boot=10_000, seed=SEED):
    rng = np.random.default_rng(seed)
    a = np.asarray(correct_after, dtype=float)
    b = np.asarray(correct_before, dtype=float)
    idx = rng.integers(0, len(a), size=(n_boot, len(a)))
    diffs = a[idx].mean(axis=1) - b[idx].mean(axis=1)
    return a.mean() - b.mean(), np.percentile(diffs, [2.5, 97.5])


def compare(before, after):
    b = np.array(before["correct"])
    a = np.array(after["correct"])
    transitions = {
        "ancora corrette": int((b & a).sum()),
        "nuove corrette": int((~b & a).sum()),
        "nuove sbagliate": int((b & ~a).sum()),
        "ancora sbagliate": int((~b & ~a).sum()),
    }
    fixed, broken = transitions["nuove corrette"], transitions["nuove sbagliate"]
    p_value = binomtest(fixed, fixed + broken, 0.5).pvalue if fixed + broken else 1.0
    diff, (low, high) = paired_bootstrap(a, b)
    return {"accuracy_before": float(b.mean()), "accuracy_after": float(a.mean()),
            "difference": float(diff), "ci95": [float(low), float(high)],
            "mcnemar_p": float(p_value), **transitions}


report = {"dev": compare(dev_before, dev_after), "holdout": compare(holdout_before, holdout_after)}
for split, r in report.items():
    print(f"\n{split.upper()}: {r['accuracy_before']:.3f} -> {r['accuracy_after']:.3f} "
          f"({r['difference']:+.3f}, IC95% [{r['ci95'][0]:+.3f}, {r['ci95'][1]:+.3f}])")
    print(f"  nuove corrette {r['nuove corrette']}, nuove sbagliate {r['nuove sbagliate']}, "
          f"McNemar p = {r['mcnemar_p']:.4f}")
save_results(report, "report.json")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

splits = list(report)
x = np.arange(len(splits))
axes[0].bar(x - 0.2, [report[s]["accuracy_before"] for s in splits], 0.4, label="prima", color="#93c5fd")
axes[0].bar(x + 0.2, [report[s]["accuracy_after"] for s in splits], 0.4, label="dopo", color="#16a34a")
axes[0].set_xticks(x, splits)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("accuratezza")
axes[0].set_title("Accuratezza prima e dopo")
axes[0].legend()

keys = ["ancora corrette", "nuove corrette", "nuove sbagliate", "ancora sbagliate"]
axes[1].bar(keys, [report["holdout"][k] for k in keys], color=["#16a34a", "#86efac", "#ea580c", "#dc2626"])
axes[1].set_title("Holdout: come cambiano le singole domande")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## 11. E adesso? Un altro giro

Il ciclo non finisce qui. Si rifà l'error analysis sugli errori **rimasti** e si guarda come è cambiata la distribuzione delle categorie: quali errori sono diminuiti, quali no, se ne sono comparsi di nuovi.

In [ ]:
dev_errors_after = analyze_errors(dev_after) if BACKEND == "hf" else [
    {"category": e["category"]} for e in dev_errors if not dev_after["correct"][e["index"]]]

before_counts = Counter(e["category"] for e in dev_errors)
after_counts = Counter(e["category"] for e in dev_errors_after)
table = pd.DataFrame({"prima": before_counts, "dopo": after_counts}).fillna(0).astype(int)
table["variazione"] = table["dopo"] - table["prima"]
print(table.sort_values("prima", ascending=False))

## Da provare

- **Selezione casuale contro selezione mirata**: con `BACKEND=hf` rifai il fine-tuning con `finetune_lora(random_baseline, "lora_random")` e confronta l'holdout. Se i dati mirati non battono quelli casuali, la selezione per similarità non sta aiutando.
- **Giudice LLM**: imposta `JUDGE_MODEL` (per esempio `Qwen/Qwen2.5-1.5B-Instruct`) e confronta le categorie con quelle delle regole.
- **Budget**: cambia `BUDGET_PER_CATEGORY` e guarda come cambia il risultato sull'holdout.

Per il contesto: [Error analysis e costruzione degli eval](https://bigghis.github.io/posts/POST-TRAINING-ERROR-ANALYSIS/) e [Post-training in pratica: migliorare un modello partendo dagli eval](https://bigghis.github.io/posts/POST-TRAINING-EVAL-LAB/).